
<div style="text-align: center; background-color: lightblue; padding: 15px;">
    <p style="text-align: center;">بسم الله، والحمد لله، والصلاة والسلام على رسول الله وعلى آله وصحبه</p>
</div>

# THIS NOTEBOOKS AIMS TO EXPLORE THE LLMs  

In [1]:
from me_r_sy.ingestion.grib import get_grib , get_var
from me_r_sy.analysis.statistics import Statistics
from me_r_sy.llm.generator import LLM , ReportGenerator
import geopandas as gpd
import numpy as np
import yaml
import calendar
import json
import subprocess

/home/muhammed/anaconda3/envs/me_r_sy/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# read the config file
config_file = "/home/muhammed/METEO-REPORTING-SYSTEM/ME_R_SY/config/config_2026_03.yml"
with open(config_file , "r") as f :
    config = yaml.safe_load(f)

In [3]:
year       = config["date"]["year"]
month      = config["date"]["month"]
input_dir  = config["data"]["input"] 
ref_dir    = f"{input_dir}/reference_1991_2020.grib"
output_dir = config["data"]["output"] 
report_dir = f"{output_dir}/report{year}_{month}"
shp_path   = config["data"]["shapefile"]
year       = config["date"]["year"]
month      = config["date"]["month"]

In [4]:
# read the shapefile
grib_path = f"{input_dir}/input_{year}_{month:02d}.grib"

ref_path  = f"{input_dir}/reference_1991_2020.grib"

cmap_dict = {"2t": "RdYlBu_r", "tp": "GnBu", "msl": "viridis", "w": "YlOrRd"}

cmap_dict_anom = {"2t": "bwr", "tp": "PuOr", "msl": "BrBG", "w": "RdGy"}

shp_path = "/home/muhammed/METEO-REPORTING-SYSTEM/ME_R_SY/data/regions/regions.shp"

shp = gpd.read_file(shp_path)
ds        = get_grib(grib_path)
ds_ref    = get_grib(ref_path)
ref_mean  = list(map(lambda ds : Statistics.ref_avg(ds , month) , ds_ref))
month_range = calendar.monthrange(year , month)[1]

/home/muhammed/anaconda3/envs/me_r_sy/lib/python3.11/site-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
/home/muhammed/anaconda3/envs/me_r_sy/lib/python3.11/site-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **

In [5]:
variables = get_var(ds)
ref_variables = get_var(ref_mean)
ref_variables["tp"] = ref_variables["tp"] * month_range

In [6]:
Stats = Statistics()
# Plot = Plotter(shp , cmap_dict , cmap_dict_anom)
statistics  = Stats.calculate_statistics(variables = variables)
daily_stats = Stats.calculate_daily(variables)
anomaly     = Stats.calculate_anomaly(statistics , ref_variables)

In [7]:
llm = LLM(daily_stats , anomaly , year , month )
generator = ReportGenerator()
message = llm.create_message()
# json_message = llm.to_json()

In [8]:
rep_gen = ReportGenerator()
sections = rep_gen.generate_report(message)

# sections["summary"]
# sections["temperature"]
# sections["precipitation"]
# sections["pressure"]

# sections["wind"]

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


RESPONSE 

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""Pour le mois de mars 2026, l'analyse des anomalies climatiques par rapport à la période de référence 1991-2020 met en évidence une variation des paramètres météorologiques. 

L'anomalie de la température à 2 mètres (t2m) s'établit à -0,74 °C. Au cours de ce mois, la température maximale a atteint 40,23 °C le 26 mars 2026 à 16:00 UTC (localisée à la longitude -0,25 et la latitude 20,0), tandis que la température minimale relevée est de 1""",
        thought_signature=b'\x12`\n^\x01i\x14}\x13y8\xcb\xd3\xf3\xa2\xd2"\xaa\xb4\xde\xc5\xc0n!:/\xb7h\x03\xf7\xc4\xce\x97/\x07d\xa7\xa9\x91\xd7^\xaf\x1b\xc5_\x10\'\x8f\xe5X\x8c\x110\xc6\xfdY\xf8>\xc1?$\x8c\xd2`5\xe1\x8cFQ\x85\xd2bx\xc8\r\xb7\x98NC\x99\x16HB\x18\x87\x06\xda9\xdc\xb7\x17u\xab2m\x1b-e'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.MAX_TOKENS: 'MAX_TOKENS'>,
  index

In [9]:
sections["summary"]
# sections["temperature"]
# sections["precipitation"]
# sections["pressure"]

# sections["wind"]

"Le mois de mars 2026 a été caractérisé par des conditions thermiques globalement inférieures aux normales de saison, affichant une anomalie de température moyenne à 2 mètres de -0,74 °C par rapport à la période de référence 1991-2020. Malgré cette tendance plus fraîche, le mois a connu d'importants contrastes thermiques, avec un pic de chaleur remarquable atteignant 40,23 °C le 26 mars. Du côté de la pression atmosphérique, un notable maximum de 1034,51 hPa a été enregistré au début du mois, le 5 mars. Enfin, les conditions de vent ont ponctuellement marqué des épisodes de dynamisme notable, illustrés notamment par une composante zonale minimale de -4,70"

In [16]:
output_dir

'/home/muhammed/METEO-REPORTING-SYSTEM/ME_R_SY/reports'

In [24]:
tex_doc = rf"""
\documentclass[a4paper,12pt]{{article}}

\usepackage[utf8]{{inputenc}}
\usepackage[T1]{{fontenc}}
\usepackage{{graphicx}}
\usepackage{{float}}
\usepackage{{geometry}}

\geometry{{margin=2.5cm}}

\title{{Rapport météorologique mensuel}}
\author{{ME-R-SY}}
\date{{}}

\begin{{document}}

\maketitle

\section{{Synthèse générale}}

{sections["summary"]}

\begin{{figure}}[H]
    \centering
    \includegraphics[width=\textwidth]{{figures/mean_{year}_{month:02d}.png}}
    \caption{{Moyennes météorologiques}}
\end{{figure}}

\section{{Conditions mensuelles}}

{sections["monthly"]}

\section{{Maximums}}

{sections["max"]}

\begin{{figure}}[H]
    \centering
    \includegraphics[width=\textwidth]{{figures/max_{year}_{month:02d}.png}}
    \caption{{Valeurs maximales}}
\end{{figure}}

\section{{Minimums}}

{sections["min"]}

\begin{{figure}}[H]
    \centering
    \includegraphics[width=\textwidth]{{figures/min_{year}_{month:02d}.png}}
    \caption{{Valeurs minimales}}
\end{{figure}}

\section{{Anomalies}}

{sections["anomaly"]}

\begin{{figure}}[H]
    \centering
    \includegraphics[width=\textwidth]{{figures/anom_{year}_{month:02d}.png}}
    \caption{{Anomalies météorologiques}}
\end{{figure}}

\end{{document}}
"""